<a href="https://colab.research.google.com/github/nrbeca/Reportes-de-presupuesto/blob/main/Validador_Pp_Partida.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pandas openpyxl -q

import pandas as pd
from google.colab import files

print(" Sube el catálogo: Pp_-_Partida_Especifica_2026.xlsx")
f = files.upload()

if f:
    df = pd.read_excel(list(f.keys())[0], header=None, dtype=str)
    df = df.iloc[1:].reset_index(drop=True)

    partidas_por_pp = {}

    for _, row in df.iterrows():
        mod = str(row.iloc[2]).strip()
        prog = str(row.iloc[4]).strip().zfill(3)
        partida = str(row.iloc[6]).strip().zfill(5)

        if mod and prog and partida and partida != 'nan':
            pp = f"{mod}{prog}"
            if pp not in partidas_por_pp:
                partidas_por_pp[pp] = set()
            partidas_por_pp[pp].add(partida)

    print(f"\n Catálogo cargado:")
    print(f"   - {len(partidas_por_pp)} Pps")
    print(f"   - {sum(len(v) for v in partidas_por_pp.values())} partidas")
    print()
    print("Pps disponibles:")
    for pp in sorted(partidas_por_pp.keys()):
        print(f"   {pp}: {len(partidas_por_pp[pp])} partidas")

 Sube el catálogo: Pp_-_Partida_Especifica_2026.xlsx


Saving Pp - Partida Especifica 2026.xlsx to Pp - Partida Especifica 2026.xlsx

 Catálogo cargado:
   - 17 Pps
   - 2122 partidas

Pps disponibles:
   B005: 217 partidas
   B006: 218 partidas
   K017: 226 partidas
   M001: 223 partidas
   O001: 206 partidas
   P021: 235 partidas
   Q004: 232 partidas
   S052: 116 partidas
   S053: 107 partidas
   S263: 27 partidas
   S290: 114 partidas
   S292: 99 partidas
   S293: 49 partidas
   S304: 34 partidas
   S318: 7 partidas
   U027: 10 partidas
   W001: 2 partidas


---\n## PASO 2: Buscar partidas válidas para un Pp

In [ ]:
#  ESCRIBE EL Pp
pp_buscar = "K017"

pp_buscar = pp_buscar.upper().strip()

if pp_buscar in partidas_por_pp:
    partidas = sorted(partidas_por_pp[pp_buscar])
    print(f" Pp {pp_buscar} tiene {len(partidas)} partidas válidas:")
    print()

    capitulos = {}
    for p in partidas:
        cap = p[0]
        if cap not in capitulos:
            capitulos[cap] = []
        capitulos[cap].append(p)

    for cap in sorted(capitulos.keys()):
        print(f"  Cap {cap}000: {', '.join(capitulos[cap])}")
else:
    print(f" Pp '{pp_buscar}' no encontrado")
    print(f"   Disponibles: {', '.join(sorted(partidas_por_pp.keys()))}")

 Pp K017 tiene 226 partidas válidas:

  Cap 1000: 11301, 12101, 12201, 12301, 13104, 13201, 13202, 13301, 13404, 13407, 13413, 14101, 14103, 14104, 14105, 14201, 14202, 14301, 14302, 14401, 14403, 14404, 14405, 14406, 15101, 15201, 15202, 15301, 15401, 15402, 15403, 15501, 15901, 16101, 16103, 16104, 16105, 16106, 16107, 16108, 17101, 17102
  Cap 2000: 21101, 21201, 21301, 21401, 21501, 21502, 21601, 22104, 22106, 22301, 24101, 24201, 24301, 24401, 24501, 24601, 24701, 24801, 24901, 25101, 25201, 25301, 25401, 25501, 25901, 26102, 26103, 26104, 26105, 27101, 27201, 27301, 27401, 27501, 29101, 29201, 29301, 29401, 29501, 29601, 29801, 29901
  Cap 3000: 31101, 31201, 31301, 31401, 31501, 31601, 31602, 31603, 31701, 31801, 31802, 31901, 31902, 31904, 32101, 32201, 32301, 32302, 32303, 32502, 32503, 32505, 32601, 32701, 32901, 33301, 33302, 33303, 33304, 33401, 33501, 33601, 33602, 33603, 33604, 33605, 33606, 33801, 33901, 33903, 34101, 34401, 34501, 34601, 34701, 35101, 35201, 35301, 3540

---\n## PASO 3: Validar una Partida específica

In [ ]:
#  ESCRIBE Pp Y PARTIDA
pp = "K017"
partida = "52301"

pp = pp.upper().strip()
partida = str(partida).strip().zfill(5)

if pp not in partidas_por_pp:
    print(f" Pp '{pp}' no existe")
elif partida in partidas_por_pp[pp]:
    print(f" VÁLIDO: Partida {partida} SÍ es válida para Pp {pp}")
else:
    print(f" INVÁLIDO: Partida {partida} NO es válida para Pp {pp}")
    cap = partida[0]
    similares = sorted([p for p in partidas_por_pp[pp] if p[0] == cap])
    if similares:
        print(f"   Partidas válidas cap {cap}000: {', '.join(similares[:15])}")

 VÁLIDO: Partida 52301 SÍ es válida para Pp K017


---\n## PASO 4: Validar archivo completo

In [5]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Border, Side

print(" Sube archivo con claves (formato PIPP o columnas Pp/Partida)")
f = files.upload()

if f:
    nombre = list(f.keys())[0]
    df_raw = pd.read_excel(nombre, header=None, dtype=str)

    # Detectar formato
    datos = []
    col_pp = None
    col_partida = None

    # Buscar fila donde empiezan los datos
    fila_datos = None
    for i in range(min(15, len(df_raw))):
        # Buscar fila que tenga un número de 1-2 dígitos en columna 0 o 1
        val0 = str(df_raw.iloc[i, 0]).strip()
        val1 = str(df_raw.iloc[i, 1]).strip() if df_raw.shape[1] > 1 else ''

        if val0.isdigit() and len(val0) <= 2 and int(val0) > 0:
            fila_datos = i
            break
        if val1.isdigit() and len(val1) <= 2 and int(val1) > 0:
            fila_datos = i
            break

    if fila_datos is not None:
        # Formato PIPP oficial
        print(f"\n✓ Formato PIPP detectado (datos desde fila {fila_datos + 1})")

        # Columnas PIPP: 0=Consec, 1=Ramo, 2=UR, 3=Año, 4=Fin, 5=Fun, 6=SF, 7=RG, 8=AI, 9=Pp, 10=Partida...
        df_datos = df_raw.iloc[fila_datos:].reset_index(drop=True)

        for _, row in df_datos.iterrows():
            pp_val = str(row.iloc[9]).strip().upper() if len(row) > 9 else ''
            partida_val = str(row.iloc[10]).strip().zfill(5) if len(row) > 10 else ''

            if pp_val and pp_val != 'nan' and partida_val and partida_val != '0000n':
                datos.append({'PP': pp_val, 'PARTIDA': partida_val})
    else:
        # Formato con columnas nombradas
        print("\n✓ Formato con columnas")
        df_cols = pd.read_excel(nombre, dtype=str)

        for c in df_cols.columns:
            cu = str(c).upper()
            if 'PP' in cu or 'PROGRAMA' in cu:
                col_pp = c
            if 'PARTIDA' in cu or 'OBJETO' in cu:
                col_partida = c

        if col_pp and col_partida:
            for _, row in df_cols.iterrows():
                pp_val = str(row[col_pp]).strip().upper()
                partida_val = str(row[col_partida]).strip().zfill(5)
                if pp_val and pp_val != 'nan':
                    datos.append({'PP': pp_val, 'PARTIDA': partida_val})
        else:
            print(f" No encontré columnas Pp y Partida")
            print(f"   Columnas: {list(df_cols.columns)}")

    if datos:
        print(f"✓ {len(datos)} registros a validar")

        # Validar
        resultados = []
        for d in datos:
            pp = d['PP']
            partida = d['PARTIDA']

            if pp not in partidas_por_pp:
                resultados.append({'PP': pp, 'PARTIDA': partida, 'VÁLIDO': 'NO', 'MOTIVO': f'Pp {pp} no existe'})
            elif partida in partidas_por_pp[pp]:
                resultados.append({'PP': pp, 'PARTIDA': partida, 'VÁLIDO': 'SI', 'MOTIVO': ''})
            else:
                resultados.append({'PP': pp, 'PARTIDA': partida, 'VÁLIDO': 'NO', 'MOTIVO': 'Partida no válida'})

        # Resumen
        validos = sum(1 for r in resultados if r['VÁLIDO'] == 'SI')
        print(f"\n{'='*50}")
        print(f"RESULTADO: {validos}/{len(resultados)} válidos")
        print(f"{'='*50}")

        errores = [r for r in resultados if r['VÁLIDO'] == 'NO']
        if errores:
            print(f"\n {len(errores)} errores:")
            for e in errores[:15]:
                print(f"   Pp={e['PP']}, Partida={e['PARTIDA']} → {e['MOTIVO']}")
            if len(errores) > 15:
                print(f"   ...y {len(errores)-15} más")

        # Exportar
        wb = Workbook()
        ws = wb.active

        si = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')
        no = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')
        bd = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))

        ws.cell(row=1, column=1, value='PP').font = Font(bold=True)
        ws.cell(row=1, column=2, value='PARTIDA').font = Font(bold=True)
        ws.cell(row=1, column=3, value='VÁLIDO').font = Font(bold=True)
        ws.cell(row=1, column=4, value='MOTIVO').font = Font(bold=True)

        for i, r in enumerate(resultados, 2):
            ws.cell(row=i, column=1, value=r['PP']).border = bd
            ws.cell(row=i, column=2, value=r['PARTIDA']).border = bd
            cell = ws.cell(row=i, column=3, value=r['VÁLIDO'])
            cell.border = bd
            cell.fill = si if r['VÁLIDO'] == 'SI' else no
            ws.cell(row=i, column=4, value=r['MOTIVO']).border = bd

        wb.save("Validacion_Pp_Partida.xlsx")
        files.download("Validacion_Pp_Partida.xlsx")
        print("\n Archivo exportado")

 Sube archivo con claves (formato PIPP o columnas Pp/Partida)


Saving Formato Estructura Programatica_2026 UR VST.xlsx to Formato Estructura Programatica_2026 UR VST.xlsx

✓ Formato PIPP detectado (datos desde fila 10)
✓ 2 registros a validar

RESULTADO: 0/2 válidos

 2 errores:
   Pp=S052, Partida=33201 → Partida no válida
   Pp=B006, Partida=33201 → Partida no válida


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 Archivo exportado


necesito este código que es un buscador validador pero en github para poder hacer una app en streamlit.io